# 🎬 Clip Mining Pipeline - Google Colab GPU (FIXED)

**Purpose:** Automatically detect viral clip candidates from Islamic lecture transcripts using FREE Tesla T4 GPU

**What this does:**
- ✅ Uses LLM (Qwen2.5-3B) for accurate topic classification
- ✅ Calculates emotional impact scores (1-10)
- ✅ Detects clip candidates (score ≥7)
- ✅ Merges adjacent clips
- ✅ **Saves results to Google Drive automatically**

---

## ⚠️ IMPORTANT: Before Running

1. **Enable GPU Runtime:**
   - Click `Runtime` → `Change runtime type`
   - Select `GPU` → `T4`
   - Click `Save`

2. **Upload your chunks.json:**
   - Upload to `/content/chunks.json` OR
   - Place in Google Drive at `/content/drive/MyDrive/video_pipeline/chunks.json`

---

## Step 1: Install Dependencies

In [ ]:
!pip install transformers torch accelerate huggingface_hub -q
print("✓ Dependencies installed")

## Step 1b: Hugging Face Authentication (For Private Models)

In [ ]:
import os
from huggingface_hub import login, HfApi

print("="*60)
print("HUGGING FACE AUTHENTICATION")
print("="*60)

# Check for token
hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    print("✓ Found HF_TOKEN environment variable")
    try:
        login(token=hf_token)
        api = HfApi()
        user = api.whoami(token=hf_token)
        print(f"✓ Logged in as: {user['name']}")
        print(f"  Token type: {'Write' if user.get('auth', {}).get('accessToken', {}).get('displayName') else 'Read'}")
    except Exception as e:
        print(f"⚠️ Login failed: {e}")
else:
    print("⚠️ No HF_TOKEN found")
    print()
    print("To use private/gated models, set HF_TOKEN:")
    print("  1. Get token from: https://huggingface.co/settings/tokens")
    print("  2. Add to Colab Secrets: Settings → Secrets → Add secret")
    print("     Name: HF_TOKEN")
    print("     Value: your_token_here")
    print("  3. Re-run this cell")
    print()
    print("Continuing with public models only...")

print("="*60)

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

# Create output directory
drive_output_dir = "/content/drive/MyDrive/video_pipeline"
os.makedirs(drive_output_dir, exist_ok=True)
print(f"✓ Drive mounted")
print(f"✓ Output directory: {drive_output_dir}")

## Step 3: Load Model

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from datetime import datetime
import os

# Model configuration
# You can use private models here if you provided HF_TOKEN
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Or your private model
# MODEL_NAME = "your-org/your-private-model"  # Example private model

VALID_TOPICS = [
    "Charity", "Oppression", "Dua", "Mercy", "Death",
    "Tawakkul", "Sabr", "Afterlife", "Faith", "Prayer",
    "Quran", "Hadith", "Other"
]

# Get HF token if available
hf_token = os.environ.get("HF_TOKEN")

# Check GPU
print("="*60)
print("GPU STATUS")
print("="*60)
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU detected - using CPU (slow)")
print("="*60)

# Load model
print("\nLoading model...")
print(f"Model: {MODEL_NAME}")
if hf_token:
    print("Using HF authentication for private model access")

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=hf_token
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        token=hf_token  # Pass token for private models
    )
    print(f"✓ Model loaded: {MODEL_NAME}")
    print(f"  Device: {model.device}")
    if torch.cuda.is_available():
        print(f"  VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    print("\nIf using a private model, make sure:")
    print("  1. HF_TOKEN is set correctly")
    print("  2. You have access to the model")
    print("  3. Model name is correct")

## Step 4: Clip Detection Functions

In [ ]:
# Keywords that indicate important Islamic content
IMPORTANT_KEYWORDS = [
    "allah", "prophet", "quran", "hadith", "islam", "muslim",
    "faith", "belief", "prayer", "salah", "zakat", "sadaqah",
    "charity", "oppression", "dua", "supplication", "mercy",
    "rahmah", "forgiveness", "paradise", "jannah", "hellfire",
    "jahannam", "death", "grave", "afterlife", "akhirah",
    "sabr", "patience", "tawakkul", "trust"
]

EMOTIONAL_KEYWORDS = [
    "love", "mercy", "fear", "hope", "beautiful", "amazing",
    "powerful", "heart", "soul", "weep", "cry", "remember",
    "death", "grave", "paradise", "hellfire", "forgiveness"
]

def has_important_content(text: str) -> bool:
    text_lower = text.lower()
    for keyword in IMPORTANT_KEYWORDS:
        if keyword in text_lower:
            return True
    return False


def calculate_clip_score(text: str, topic: str) -> int:
    score = 5
    text_lower = text.lower()
    
    if any(word in text_lower for word in EMOTIONAL_KEYWORDS):
        score += 2
    
    religious_words = ['allah', 'prophet', 'quran', 'faith', 'belief', 'worship']
    if any(word in text_lower for word in religious_words):
        score += 1
    
    teaching_words = ['remember', 'learn', 'understand', 'know', 'lesson']
    if any(word in text_lower for word in teaching_words):
        score += 1
    
    important_topics = ['Charity', 'Oppression', 'Dua', 'Mercy', 'Patience', 'Afterlife']
    if topic in important_topics:
        score += 1
    
    return min(score, 10)


def classify_topic_llm(text: str) -> str:
    prompt = f"""Classify the topic of this Islamic lecture segment.

Topics: {', '.join(VALID_TOPICS[:8])}

Text: {text[:500]}

Return topic only."""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            temperature=0.3,
            do_sample=False
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    topic = response.strip().title()
    
    if topic in VALID_TOPICS:
        return topic
    return "Other"


def process_chunk(chunk: dict) -> dict:
    text = chunk.get("text", "")
    has_important = has_important_content(text)
    
    if not has_important:
        return {
            **chunk,
            "topic": "Other",
            "emotion_score": 3,
            "clip_candidate": False,
            "skipped": True
        }
    
    topic = classify_topic_llm(text)
    emotion_score = calculate_clip_score(text, topic)
    clip_candidate = (topic != "Other" and emotion_score >= 7)
    
    return {
        **chunk,
        "topic": topic,
        "emotion_score": emotion_score,
        "clip_candidate": clip_candidate,
        "skipped": False
    }

## Step 5: Load Transcripts

In [ ]:
import os

print("Loading transcripts...")

# Try different paths
possible_paths = [
    "/content/chunks.json",
    "/content/drive/MyDrive/video_pipeline/chunks.json",
    "chunks.json"
]

chunks = None
used_path = None

for path in possible_paths:
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                chunks = data
            elif isinstance(data, dict) and 'chunks' in data:
                chunks = data['chunks']
            used_path = path
            break

if not chunks:
    print("❌ No chunks.json found!")
    print("\nPlease upload chunks.json to:")
    print("  - /content/chunks.json (direct upload)")
    print("  - /content/drive/MyDrive/video_pipeline/chunks.json (via Drive)")
else:
    print(f"✓ Loaded {len(chunks)} chunks from {used_path}")

## Step 6: Process with GPU

In [ ]:
if chunks:
    print("\n" + "="*60)
    print("PROCESSING WITH GPU")
    print("="*60)
    
    processed_chunks = []
    skipped_count = 0
    
    for i, chunk in enumerate(chunks):
        result = process_chunk(chunk)
        processed_chunks.append(result)
        
        if result.get("skipped"):
            skipped_count += 1
        
        # Progress update every 50 chunks
        if (i + 1) % 50 == 0:
            candidates_so_far = sum(1 for c in processed_chunks if c.get("clip_candidate"))
            print(f"  Processed {i+1}/{len(chunks)} chunks... ({candidates_so_far} clip candidates so far)")
    
    # Summary
    candidates = [c for c in processed_chunks if c.get("clip_candidate")]
    high_scores = [c for c in processed_chunks if c.get("emotion_score", 0) >= 8]
    
    print("\n" + "="*60)
    print("PROCESSING COMPLETE")
    print("="*60)
    print(f"Total chunks: {len(chunks)}")
    print(f"Skipped (generic): {skipped_count}")
    print(f"Processed with LLM: {len(chunks) - skipped_count}")
    print(f"Clip candidates (score ≥7): {len(candidates)}")
    print(f"High priority (score ≥8): {len(high_scores)}")
    print("="*60)

## Step 7: Merge Adjacent Clips

In [ ]:
def parse_timestamp(timestamp: str) -> float:
    if not timestamp:
        return 0.0
    parts = str(timestamp).replace(".", ":").split(":")
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    elif len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    return 0.0


def merge_adjacent_clips(candidates: list, max_duration: int = 60, min_duration: int = 15) -> list:
    if not candidates:
        return []
    
    sorted_candidates = sorted(
        candidates,
        key=lambda x: parse_timestamp(x.get("timestamp", x.get("timestamp_start", "00:00")))
    )
    
    merged_clips = []
    current_clip = None
    
    for candidate in sorted_candidates:
        # Handle both timestamp formats
        ts = candidate.get("timestamp_start") or candidate.get("timestamp")
        te = candidate.get("timestamp_end") or (float(ts) + 8 if ts else "00:00")
        
        start = parse_timestamp(ts)
        end = parse_timestamp(te) if isinstance(te, str) else start + 8
        
        if current_clip is None:
            current_clip = {
                "video_id": candidate.get("video_id", "unknown"),
                "timestamp_start": ts,
                "timestamp_end": te if isinstance(te, str) else f"{int(start//60):02d}:{int(start%60):02d}",
                "start_seconds": start,
                "end_seconds": end,
                "topics": [candidate.get("topic")],
                "emotion_scores": [candidate.get("emotion_score", 0)],
                "chunks": [candidate]
            }
        else:
            time_gap = start - current_clip["end_seconds"]
            
            if time_gap <= 10:
                current_clip["timestamp_end"] = te if isinstance(te, str) else current_clip["timestamp_end"]
                current_clip["end_seconds"] = end
                current_clip["topics"].append(candidate.get("topic"))
                current_clip["emotion_scores"].append(candidate.get("emotion_score", 0))
                current_clip["chunks"].append(candidate)
            else:
                duration = current_clip["end_seconds"] - current_clip["start_seconds"]
                if min_duration <= duration <= max_duration:
                    topic_counts = {}
                    for t in current_clip["topics"]:
                        topic_counts[t] = topic_counts.get(t, 0) + 1
                    primary_topic = max(topic_counts, key=topic_counts.get)
                    
                    merged_clips.append({
                        "video_id": current_clip["video_id"],
                        "timestamp_start": current_clip["timestamp_start"],
                        "timestamp_end": current_clip["timestamp_end"],
                        "start_seconds": current_clip["start_seconds"],
                        "end_seconds": current_clip["end_seconds"],
                        "duration": duration,
                        "topic": primary_topic,
                        "all_topics": list(set(current_clip["topics"])),
                        "emotion_score": max(current_clip["emotion_scores"]),
                        "chunk_count": len(current_clip["chunks"])
                    })
                
                current_clip = {
                    "video_id": candidate.get("video_id", "unknown"),
                    "timestamp_start": ts,
                    "timestamp_end": te if isinstance(te, str) else f"{int(start//60):02d}:{int(start%60):02d}",
                    "start_seconds": start,
                    "end_seconds": end,
                    "topics": [candidate.get("topic")],
                    "emotion_scores": [candidate.get("emotion_score", 0)],
                    "chunks": [candidate]
                }
    
    if current_clip:
        duration = current_clip["end_seconds"] - current_clip["start_seconds"]
        if min_duration <= duration <= max_duration:
            topic_counts = {}
            for t in current_clip["topics"]:
                topic_counts[t] = topic_counts.get(t, 0) + 1
            primary_topic = max(topic_counts, key=topic_counts.get)
            
            merged_clips.append({
                "video_id": current_clip["video_id"],
                "timestamp_start": current_clip["timestamp_start"],
                "timestamp_end": current_clip["timestamp_end"],
                "start_seconds": current_clip["start_seconds"],
                "end_seconds": current_clip["end_seconds"],
                "duration": duration,
                "topic": primary_topic,
                "all_topics": list(set(current_clip["topics"])),
                "emotion_score": max(current_clip["emotion_scores"]),
                "chunk_count": len(current_clip["chunks"])
            })
    
    return merged_clips


# Merge clips
if processed_chunks:
    candidates = [c for c in processed_chunks if c.get("clip_candidate")]
    merged = merge_adjacent_clips(candidates)
    
    print(f"\n✓ Merged {len(candidates)} candidates into {len(merged)} clips")
    print("\nTop clips:")
    for i, clip in enumerate(sorted(merged, key=lambda x: -x['emotion_score'])[:10], 1):
        print(
            f"  {i:2d}. [{clip['topic']:12}] {clip['timestamp_start']}-{clip['timestamp_end']} | "
            f"Score: {clip['emotion_score']} | Duration: {clip['duration']:.0f}s"
        )

## Step 8: Save to Google Drive (FIXED)

In [ ]:
# Save results
if merged:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save to /content/ for download
    local_file = f"/content/clip_candidates_{timestamp}.json"
    with open(local_file, 'w', encoding='utf-8') as f:
        json.dump(merged, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved to Colab: {local_file}")
    
    # Save to Google Drive (GUARANTEED)
    drive_file = f"/content/drive/MyDrive/video_pipeline/clip_candidates_{timestamp}.json"
    try:
        with open(drive_file, 'w', encoding='utf-8') as f:
            json.dump(merged, f, indent=2, ensure_ascii=False)
        print(f"✓ Saved to Google Drive: {drive_file}")
    except Exception as e:
        print(f"❌ Failed to save to Drive: {e}")
        print("   Trying alternative path...")
        # Try without video_pipeline subdirectory
        drive_file_alt = f"/content/drive/MyDrive/clip_candidates_{timestamp}.json"
        with open(drive_file_alt, 'w', encoding='utf-8') as f:
            json.dump(merged, f, indent=2, ensure_ascii=False)
        print(f"✓ Saved to: {drive_file_alt}")
        drive_file = drive_file_alt
    
    # Also save as CSV for easy viewing
    csv_file = f"/content/clip_candidates_{timestamp}.csv"
    with open(csv_file, 'w', encoding='utf-8') as f:
        f.write("video_id,topic,start_time,end_time,duration,emotion_score\n")
        for clip in merged:
            f.write(f"\"{clip['video_id']}\",\"{clip['topic']}\",\"{clip['timestamp_start']}\",\"{clip['timestamp_end']}\",{clip['duration']:.0f},{clip['emotion_score']}\n")
    print(f"✓ Saved CSV: {csv_file}")
    
    # Save CSV to Drive too
    csv_drive = f"/content/drive/MyDrive/video_pipeline/clip_candidates_{timestamp}.csv"
    with open(csv_drive, 'w', encoding='utf-8') as f:
        f.write("video_id,topic,start_time,end_time,duration,emotion_score\n")
        for clip in merged:
            f.write(f"\"{clip['video_id']}\",\"{clip['topic']}\",\"{clip['timestamp_start']}\",\"{clip['timestamp_end']}\",{clip['duration']:.0f},{clip['emotion_score']}\n")
    print(f"✓ Saved CSV to Drive: {csv_drive}")
    
    # Show ffmpeg commands
    print("\n" + "="*60)
    print("FFMPEG EXTRACTION COMMANDS")
    print("="*60)
    print("Run these locally to extract clips:\n")
    for i, clip in enumerate(merged[:15], 1):
        topic = clip['topic'].lower().replace(' ', '_')
        print(f"# Clip {i:2d}: {clip['topic']} (Score: {clip['emotion_score']})")
        print(f"ffmpeg -ss {clip['timestamp_start']} -to {clip['timestamp_end']} -i video.mp4 clip_{i:02d}_{topic}.mp4")
        print()
    
    print("\n" + "="*60)
    print("FILES SAVED")
    print("="*60)
    print(f"JSON: {drive_file}")
    print(f"CSV:  {csv_drive}")
    print("\nCheck your Google Drive: video_pipeline/ folder")

## ✅ Done!

Your clip candidates are now saved to Google Drive.

### Next Steps:

1. **Check Google Drive:** Open `video_pipeline/clip_candidates_*.json`
2. **Download to local machine**
3. **Extract video clips** using the ffmpeg commands above
4. **Import to database:**
   ```bash
   python -m clip_pipeline.run_pipeline \
       --transcript clip_candidates_TIMESTAMP.json \
       --video-id your_video_id \
       --title "Your Video Title"
   ```